In [12]:
import holidays
import pandas as pd

In [13]:
# Load in dataframe
df = pd.read_parquet('../data/processed/electricity_transformed.parquet')

# Checking date range
print(df['timestamp'].iloc[0])
print(df['timestamp'].iloc[-1])

2020-09-15 00:00:00+10:00
2026-09-23 23:00:00+10:00


In [14]:
# Fetch NSW holidays
nsw_hols = holidays.Australia(subdiv='NSW', years=range(2020, 2027))

# Add new columns for date, weekday, and if it's a holiday
df['date'] = df['timestamp'].dt.date
df['weekday'] = df['timestamp'].dt.day_name()
df['is_holiday'] = df['date'].isin(nsw_hols)

In [17]:
# Average daily demand
daily = df.groupby(['date', 'weekday', 'is_holiday'])['demand'].mean().reset_index()
daily.head(7)

,date,weekday,is_holiday,demand
0,2020-09-15,Tuesday,False,7314.456910
1,2020-09-16,Wednesday,False,7164.403299
2,2020-09-17,Thursday,False,7202.918924
3,2020-09-18,Friday,False,7520.126667
4,2020-09-19,Saturday,False,6794.693993
5,2020-09-20,Sunday,False,6793.199549
6,2020-09-21,Monday,False,7196.622847


In [18]:
order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Holiday vs normal day
comp = daily.pivot_table(index='weekday', columns='is_holiday', values='demand', aggfunc='mean').reindex(order)
comp.columns = ['normal', 'holiday']
comp['diff_%'] = (comp['holiday'] / comp['normal'] - 1) * 100
comp

,normal,holiday,diff_%
weekday,,,
Monday,7688.660271,6889.180639,-10.398166
Tuesday,7729.030838,6976.535472,-9.735960
Wednesday,7702.260009,6490.508750,-15.732412
Thursday,7773.033643,6587.811655,-15.247869
Friday,7720.069645,6526.157576,-15.465043
Saturday,7216.037381,6481.073188,-10.185149
Sunday,7052.289384,6316.461378,-10.433888


In [19]:
# Each holiday vs normal days on same weekday
hol_days = daily[daily['is_holiday']].copy()
hol_days['name'] = hol_days['date'].map(nsw_hols.get)
hol_days['weekday_avg'] = hol_days['weekday'].map(comp['normal'])
hol_days['diff_%'] = (hol_days['demand'] / hol_days['weekday_avg'] - 1) * 100
hol_days.sort_values('diff_%')

,date,weekday,is_holiday,demand,name,weekday_avg,diff_%
1927,2025-12-25,Thursday,True,5540.804514,Christmas Day,7773.033643,-28.717605
1562,2024-12-25,Wednesday,True,5551.395174,Christmas Day,7702.260009,-27.925113
1483,2024-10-07,Monday,True,5778.897882,Labor Day,7688.660271,-24.838689
1928,2025-12-26,Friday,True,5868.095070,Boxing Day,7720.069645,-23.989092
1196,2023-12-25,Monday,True,5952.870313,Christmas Day,7688.660271,-22.575974
...,...,...,...,...,...,...,...
272,2021-06-14,Monday,True,8255.818090,Queen's Birthday,7688.660271,7.376549
1228,2024-01-26,Friday,True,8356.983160,Australia Day,7720.069645,8.250101
636,2022-06-13,Monday,True,8433.390069,Queen's Birthday,7688.660271,9.686080
1728,2025-06-09,Monday,True,8861.959722,King's Birthday,7688.660271,15.260129
